# Concept Extraction from Jeopardy Clues

Extract a cultural literacy concept list from 530k Jeopardy clues using:
1. **Response-based concepts** — Normalize and deduplicate correct responses, classify entity types via NER
2. **Clue-text concepts** — Run NER on clue text to extract additional referenced entities
3. **Importance scoring** — Combine frequency, seasonal spread, category breadth into an importance metric

In [ ]:
import pandas as pd
import numpy as np
import spacy
from collections import Counter, defaultdict
import matplotlib.pyplot as plt
import seaborn as sns
import re
import warnings
warnings.filterwarnings("ignore")

sns.set_style("whitegrid")
plt.rcParams["figure.figsize"] = (12, 6)

nlp = spacy.load("en_core_web_sm")
print(f"spaCy model loaded: {nlp.meta['name']} v{nlp.meta['version']}")

## 1. Load Data

In [ ]:
df = pd.read_csv(
    "../data/jeopardy_clue_dataset/combined_season1-41.tsv",
    sep="\t",
    dtype={"clue_value": "Int64", "daily_double_value": "Int64"},
    parse_dates=["air_date"],
    low_memory=False,
)

# Derive season
df["season"] = df["air_date"].apply(lambda d: d.year if d.month >= 9 else d.year - 1)

# Drop the 2 rows with null responses
df = df.dropna(subset=["question"])

print(f"Loaded {len(df):,} clues")
print(f"Seasons: {df['season'].min()} to {df['season'].max()}")

## 2. Response Normalization

The correct responses are already entity-like (median 2 words). Normalize them by:
- Lowercasing
- Stripping leading articles (the, a, an)
- Stripping common Jeopardy prefixes ("What is", "Who is", etc.)
- Collapsing whitespace

In [ ]:
def normalize_response(text):
    """Normalize a Jeopardy response to a canonical concept form."""
    if not isinstance(text, str):
        return ""
    s = text.lower().strip()
    # Remove Jeopardy phrasing if present
    s = re.sub(r"^(what|who|where|when)\s+(is|are|was|were|does)\s+", "", s)
    # Strip leading articles
    s = re.sub(r"^(the|a|an)\s+", "", s)
    # Collapse whitespace
    s = re.sub(r"\s+", " ", s).strip()
    return s

df["concept"] = df["question"].apply(normalize_response)

# Show examples of normalization
samples = df[["question", "concept"]].drop_duplicates("question").sample(10, random_state=42)
print("Normalization examples:")
for _, row in samples.iterrows():
    print(f"  {row['question']!r:40s} -> {row['concept']!r}")

n_unique = df["concept"].nunique()
print(f"\nUnique normalized concepts: {n_unique:,}")

## 3. NER on Correct Responses

Run spaCy NER on the unique responses to classify entity types (PERSON, GPE, ORG, WORK_OF_ART, etc.). Since many responses are short names, we process them in bulk.

In [ ]:
# Get unique concepts and their frequencies
concept_counts = df["concept"].value_counts()
unique_concepts = concept_counts.index.tolist()
print(f"Running NER on {len(unique_concepts):,} unique concepts...")

# Process in batches with nlp.pipe for speed
concept_entity_types = {}
batch_size = 1000

for i in range(0, len(unique_concepts), batch_size):
    batch = unique_concepts[i:i+batch_size]
    for doc, concept in zip(nlp.pipe(batch, disable=["parser", "lemmatizer"]), batch):
        if doc.ents:
            # Take the entity type of the longest entity span
            best_ent = max(doc.ents, key=lambda e: len(e.text))
            concept_entity_types[concept] = best_ent.label_
        else:
            concept_entity_types[concept] = "UNKNOWN"
    
    if (i // batch_size) % 20 == 0:
        print(f"  Processed {min(i+batch_size, len(unique_concepts)):,} / {len(unique_concepts):,}")

print(f"\nDone. Entity type distribution:")
type_dist = Counter(concept_entity_types.values())
for t, c in type_dist.most_common():
    print(f"  {t:20s} {c:>8,}")

## 4. Build Concept Table

For each unique concept, compute:
- **frequency**: total times it was a correct response
- **seasons_appeared**: number of distinct seasons (max 41)
- **category_breadth**: number of distinct categories it appeared in
- **entity_type**: from NER classification
- **avg_clue_value**: average dollar value when this was the answer

In [ ]:
concept_stats = df.groupby("concept").agg(
    frequency=("concept", "size"),
    seasons_appeared=("season", "nunique"),
    category_breadth=("category", "nunique"),
    avg_clue_value=("clue_value", "mean"),
    first_seen=("air_date", "min"),
    last_seen=("air_date", "max"),
).reset_index()

# Add entity type
concept_stats["entity_type"] = concept_stats["concept"].map(concept_entity_types)

# Sort by frequency
concept_stats = concept_stats.sort_values("frequency", ascending=False).reset_index(drop=True)

print(f"Concept table: {len(concept_stats):,} unique concepts")
print(f"\nTop 30 concepts:")
print(concept_stats.head(30)[["concept", "frequency", "seasons_appeared", "category_breadth", "entity_type"]].to_string())

## 5. Importance Scoring

Combine metrics into a single importance score. The idea:
- **Frequency** matters, but with diminishing returns (log scale)
- **Seasonal spread** rewards durability — a concept in 40/41 seasons is more important than one in 5
- **Category breadth** rewards cross-domain relevance — Shakespeare appearing in Literature, Theater, History, and Quotes is more important than a concept that only shows up in one domain

All components are normalized to [0, 1] then combined.

In [ ]:
# Normalize each component to [0, 1]
def min_max_normalize(series):
    return (series - series.min()) / (series.max() - series.min())

concept_stats["freq_score"] = min_max_normalize(np.log1p(concept_stats["frequency"]))
concept_stats["season_score"] = min_max_normalize(concept_stats["seasons_appeared"])
concept_stats["breadth_score"] = min_max_normalize(np.log1p(concept_stats["category_breadth"]))

# Weighted combination
# Frequency: 40%, Seasonal spread: 35%, Category breadth: 25%
concept_stats["importance"] = (
    0.40 * concept_stats["freq_score"] +
    0.35 * concept_stats["season_score"] +
    0.25 * concept_stats["breadth_score"]
)

concept_stats = concept_stats.sort_values("importance", ascending=False).reset_index(drop=True)
concept_stats.index += 1  # 1-based rank
concept_stats.index.name = "rank"

print("Top 50 concepts by importance score:")
print(concept_stats.head(50)[
    ["concept", "entity_type", "frequency", "seasons_appeared", "category_breadth", "importance"]
].to_string())

In [ ]:
# Visualize top 40 by importance
top40 = concept_stats.head(40)

fig, ax = plt.subplots(figsize=(12, 10))
colors = {
    "GPE": "#2196F3", "PERSON": "#FF9800", "ORG": "#4CAF50", 
    "NORP": "#9C27B0", "LOC": "#00BCD4", "WORK_OF_ART": "#F44336",
    "EVENT": "#795548", "UNKNOWN": "#9E9E9E",
}
bar_colors = [colors.get(t, "#9E9E9E") for t in top40["entity_type"]]

ax.barh(range(len(top40)), top40["importance"], color=bar_colors, edgecolor="none")
ax.set_yticks(range(len(top40)))
ax.set_yticklabels([f"{c} ({t})" for c, t in zip(top40["concept"], top40["entity_type"])], fontsize=9)
ax.invert_yaxis()
ax.set_xlabel("Importance Score")
ax.set_title("Top 40 Cultural Literacy Concepts from Jeopardy (by Importance Score)")

# Legend
from matplotlib.patches import Patch
legend_items = [Patch(facecolor=c, label=l) for l, c in colors.items() if l in top40["entity_type"].values]
ax.legend(handles=legend_items, loc="lower right", title="Entity Type")

plt.tight_layout()
plt.show()

## 6. Entity Type Breakdown

How do the top concepts break down by type? This reveals whether the list is balanced or skewed toward geography.

In [ ]:
# Entity type distribution for top 500 concepts
top500 = concept_stats.head(500)
type_breakdown = top500["entity_type"].value_counts()
print("Entity type distribution (top 500 concepts):")
print(type_breakdown.to_string())

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Pie chart
type_breakdown.plot(kind="pie", ax=axes[0], autopct="%1.0f%%", colors=[colors.get(t, "#9E9E9E") for t in type_breakdown.index])
axes[0].set_title("Entity Type Distribution (Top 500 Concepts)")
axes[0].set_ylabel("")

# Top 10 per entity type
for etype in ["PERSON", "GPE", "ORG", "WORK_OF_ART"]:
    subset = concept_stats[concept_stats["entity_type"] == etype].head(10)
    if len(subset) > 0:
        print(f"\nTop 10 {etype}:")
        print(subset[["concept", "frequency", "importance"]].to_string())

axes[1].axis("off")
plt.tight_layout()
plt.show()

## 7. NER on Clue Text (Sample)

Run NER on the clue text (`answer` column) to extract concepts that are *referenced* in clues but aren't the correct response. This captures contextual knowledge — e.g., a clue about the Missouri River might reference Lewis & Clark, the Louisiana Purchase, and the Mississippi.

We sample 50k clues for speed, then extrapolate.

In [ ]:
# Sample 50k clues for clue-text NER
sample = df.sample(50000, random_state=42)
print(f"Running NER on {len(sample):,} clue texts...")

# Entity types to keep (skip numeric types)
KEEP_TYPES = {"PERSON", "GPE", "ORG", "LOC", "NORP", "FAC", "WORK_OF_ART", "EVENT", "PRODUCT", "LAW", "LANGUAGE"}

clue_entities = Counter()
clue_entity_types = {}

texts = sample["answer"].tolist()
for i, doc in enumerate(nlp.pipe(texts, disable=["lemmatizer"], batch_size=500)):
    for ent in doc.ents:
        if ent.label_ in KEEP_TYPES:
            normalized = normalize_response(ent.text)
            if len(normalized) > 1:  # skip single chars
                clue_entities[normalized] += 1
                clue_entity_types[normalized] = ent.label_
    if (i + 1) % 10000 == 0:
        print(f"  Processed {i+1:,} / {len(texts):,}")

print(f"\nExtracted {len(clue_entities):,} unique entities from clue text")
print(f"\nTop 30 entities mentioned in clue text:")
for entity, count in clue_entities.most_common(30):
    etype = clue_entity_types.get(entity, "?")
    print(f"  {entity:30s} {etype:15s} {count:>5}")

## 8. Combined Concept Universe

Merge response-based concepts with clue-text entities to get the full concept universe. Concepts that appear as *both* answers and clue-text references are especially important.

In [ ]:
# Mark which concepts also appear in clue text
concept_stats["in_clue_text"] = concept_stats["concept"].isin(clue_entities)
concept_stats["clue_text_mentions"] = concept_stats["concept"].map(clue_entities).fillna(0).astype(int)

# Concepts that are only in clue text (never an answer)
answer_concepts = set(concept_stats["concept"])
clue_only = {e: c for e, c in clue_entities.items() if e not in answer_concepts}

print(f"Concepts as answers: {len(answer_concepts):,}")
print(f"Concepts in clue text (from 50k sample): {len(clue_entities):,}")
print(f"Overlap (both answer and in clue text): {concept_stats['in_clue_text'].sum():,}")
print(f"Clue-text only (never an answer): {len(clue_only):,}")

print(f"\nTop 20 clue-text-only concepts (potential gaps in response-based list):")
for entity, count in sorted(clue_only.items(), key=lambda x: -x[1])[:20]:
    etype = clue_entity_types.get(entity, "?")
    print(f"  {entity:30s} {etype:15s} {count:>5}")

## 9. Export

Save the concept table for downstream use.

In [ ]:
# Save full concept table
output_path = "../data/jeopardy_concepts.csv"
export_cols = [
    "concept", "entity_type", "frequency", "seasons_appeared", 
    "category_breadth", "avg_clue_value", "first_seen", "last_seen",
    "importance", "in_clue_text", "clue_text_mentions",
]
concept_stats[export_cols].to_csv(output_path, index=True)
print(f"Saved {len(concept_stats):,} concepts to {output_path}")

# Summary stats
print(f"\n--- Summary ---")
print(f"Total unique concepts: {len(concept_stats):,}")
print(f"Concepts appearing 5+ times: {(concept_stats['frequency'] >= 5).sum():,}")
print(f"Concepts appearing 10+ times: {(concept_stats['frequency'] >= 10).sum():,}")
print(f"Concepts in 20+ seasons: {(concept_stats['seasons_appeared'] >= 20).sum():,}")
print(f"Concepts in all 41 seasons: {(concept_stats['seasons_appeared'] == 41).sum():,}")

## Next Steps

1. **Entity linking** — Map concepts to Wikidata IDs for canonical deduplication (e.g., "FDR" and "Franklin Roosevelt" -> same entity)
2. **Category-informed typing** — Use category names to improve entity classification ("PRESIDENTS" -> PERSON)
3. **Full clue-text NER** — Run on all 530k clues (not just 50k sample) for complete coverage
4. **Gap analysis vs. Hirsch** — Compare our concept list against the original 5,000 cultural literacy items
5. **Media matching** — Begin scoring books/films/shows against the weighted concept list